In [ ]:
# Parameter cell

# Model validation. Offline evaluation

In [ ]:
# Importing the libraries
from IPython.display import Markdown
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
import scipy.stats as st

import validationlib
import validationlib.models.validation

In [ ]:
if model_name is not None:
    display(Markdown(f"## Model: {model_name}"))

In [ ]:
# Load dataframes

x_train_df = pd.read_csv(os.path.join(csv_dir, "x_train.csv"))
x_test_df = pd.read_csv(os.path.join(csv_dir, "x_test.csv"))
x_val_df = pd.read_csv(os.path.join(csv_dir, "x_val.csv"))

yt_train_df = pd.read_csv(os.path.join(csv_dir, "yt_train.csv"))
yt_test_df = pd.read_csv(os.path.join(csv_dir, "yt_test.csv"))
yt_val_df = pd.read_csv(os.path.join(csv_dir, "yt_val.csv"))

yh_val_df = pd.read_csv(os.path.join(csv_dir, "yh_val.csv"))
yh_test_df = pd.read_csv(os.path.join(csv_dir, "yh_test.csv"))

In [ ]:
# Subsample the data (use only n random examples from each set)
if subsample_size is not None:
    n = subsample_size
    display(Markdown(f"### *Subsampling to {n} examples per set*"))
    train_indices = np.random.choice(x_train_df.index, n, replace=False)
    test_indices = np.random.choice(x_test_df.index, n, replace=False)
    val_indices = np.random.choice(x_val_df.index, n, replace=False)

    x_train_df = x_train_df.loc[train_indices]
    x_test_df = x_test_df.loc[test_indices]
    x_val_df = x_val_df.loc[val_indices]

    yt_train_df = yt_train_df.loc[train_indices]
    yt_test_df = yt_test_df.loc[test_indices]
    yt_val_df = yt_val_df.loc[val_indices]

    yh_val_df = yh_val_df.loc[val_indices]
    yh_test_df = yh_test_df.loc[test_indices]

In [ ]:
if splitting_variables is None:
    splitting_variables = x_train_df.columns.tolist()

In [ ]:
input_names = x_train_df.columns
output_names = yt_train_df.columns

In [ ]:
# # Get all the test data that has the same splitting variables as the training data and remove it from the test set
# used_test_indices = ~x_test_df.set_index(splitting_variables).index.isin(x_train_df.set_index(splitting_variables).index)
# used_val_indices = ~x_val_df.set_index(splitting_variables).index.isin(x_train_df.set_index(splitting_variables).index)

# x_test_df = x_test_df[used_test_indices]
# yt_test_df = yt_test_df[used_test_indices]
# yh_test_df = yh_test_df[used_test_indices]

# x_val_df = x_val_df[used_val_indices]
# yt_val_df = yt_val_df[used_val_indices]
# yh_val_df = yh_val_df[used_val_indices]

In [ ]:
spvar_train = np.array(list(x_train_df.groupby(splitting_variables).groups.keys()))
spvar_test = np.array(list(x_test_df.groupby(splitting_variables).groups.keys()))
spvar_val = np.array(list(x_val_df.groupby(splitting_variables).groups.keys()))

if len(splitting_variables) == 2:
    fig, axs = plt.subplots(figsize=(10,10))
    axs.scatter(spvar_train[:, 0], spvar_train[:, 1], label="Train", facecolors="none", edgecolors="tab:blue", s=100)
    axs.scatter(spvar_test[:, 0], spvar_test[:, 1], label="Test", marker="+", s=100, c="tab:orange")
    axs.scatter(spvar_val[:, 0], spvar_val[:, 1], label="Validation", marker="x", s=100, c="tab:green")
    axs.set_xlabel(splitting_variables[0])
    axs.set_ylabel(splitting_variables[1])
    plt.legend()
    plt.show()

---

## DATA

In this section we perform a basic exploratory analysis of the data using the whole dataset.

In [ ]:
# Since the data is already splited in training, validation and test, we merge all the subsets for this analysis
x_df = pd.concat([x_train_df, x_val_df, x_test_df], axis=0)
yt_df = pd.concat([yt_train_df, yt_val_df, yt_test_df], axis=0)

#### INPUTS

In [ ]:
statistics = {
    "mean": np.mean,
    "median": [np.percentile, {"q": 50}],
    "std": np.std,
    "IQR": st.iqr,
    "kurtosis": st.kurtosis,
    "skewness": st.skew
}
displayed_table = validationlib.tables.summary.prediction_stats(
    x_df, # Metric from which the statistics are calculated
    statistics=statistics, # Dictionary of statistics to calculate
    precision=3, # Number of decimal places to display in the table
    nsim=100, # Number of simulations for the bootstrap method
    method='Bootstrap' # Method used to calculate the confidence intervals. 
)
display(displayed_table)

validationlib.plots.histogram(
    x_df,
    xlabel="Input Variable",
    trimStds=3,
    multiPlotsKwargs={"tight_layout": True},
    logscale=True
)
plt.show()

statistics_percentile = {"%i%%" % i: [np.percentile, {"q": i}] for i in [1,5,10,50,90,95,99]} # Dict of percentiles to calculate
displayed_table = validationlib.tables.summary.prediction_stats(
    x_df,
    statistics=statistics_percentile, 
    precision=3, 
    nsim=100,
    method='Bootstrap', 
    countMinMax=False
)
display(displayed_table)

validationlib.plots.cumulative(
    x_df,
    xlabel="Ground truth", 
    bins=1000,
    quantiles=[0.1, 0.50, 0.90, 0.95, 0.99] # Quantiles to display in the plot
)
plt.show()

#### OUTPUTS

In [ ]:
statistics = {
    "mean": np.mean,
    "median": [np.percentile, {"q": 50}],
    "std": np.std,
    "IQR": st.iqr,
    "kurtosis": st.kurtosis,
    "skewness": st.skew
}
displayed_table = validationlib.tables.summary.prediction_stats(
    yt_df, # Metric from which the statistics are calculated
    statistics=statistics, # Dictionary of statistics to calculate
    precision=3, # Number of decimal places to display in the table
    nsim=100, # Number of simulations for the bootstrap method
    method='Bootstrap' # Method used to calculate the confidence intervals. 
)
display(displayed_table)
validationlib.plots.histogram(
    yt_df,
    xlabel="Output Variable",
    multiPlotsKwargs={"tight_layout": True},
    logscale=True
)
plt.show()

statistics_percentile = {"%i%%" % i: [np.percentile, {"q": i}] for i in [1,5,10,50,90,95,99]} # Dict of percentiles to calculate
displayed_table = validationlib.tables.summary.prediction_stats(
    yt_df,
    statistics=statistics_percentile, 
    precision=3, 
    nsim=100,
    method='Bootstrap', 
    countMinMax=False
)
display(displayed_table)

validationlib.plots.cumulative(
    yt_df,
    xlabel="Ground truth", 
    bins=1000,
    quantiles=[0.1, 0.50, 0.90, 0.95, 0.99] # Quantiles to display in the plot
)
plt.show()

---

## TRAIN-TEST SPLIT

To check if the splitting in train and test is correctly done, we compare the distributions of inputs and outputs between the training and test set. The empirical distributions of training and test should resemble eachother (as we want the results obtained from the test set to be representative of the performance of the model), without being too similar (as there would be risk of p-hacking: the results obtained from the test set would not representative of the model's inference power, as it would have been tested in data excesively similar to the data it has been trained on).

To compare these distributions we:
- Plot a double histogram. The histogram of the test set should look alike the one of the training set.
- Perform hypothesis tests. Specificallly, we perform a two-sample Kolmogorov-Smirnov (KS) and Anderson-Darling (AD) tests. These tests consider as null hypothesis that both datasets come from the same distribution (and as alternative, that they do not).

In [ ]:
# Train-test split
validationlib.plots.doubleHistogram(
    yt_train_df, 
    yt_test_df, 
    xlabel='Ouptut',
    x1label='Train',
    x2label='Test',
    multiPlotsKwargs = {"tight_layout":True},
    logscale=True
)
plt.show()

table_pval_num = validationlib.tests.dist.dist_similarity_table(
    yt_train_df,
    yt_test_df,
    title="p-values",
    tests=["KS", "AD"]
)
display(table_pval_num)

In [ ]:
display(Markdown(f"Since the splitting is done by the variables {splitting_variables}, the TT split check should be done by the same variables"))

x_train_spvar_df = pd.DataFrame(x_train_df.groupby(splitting_variables).groups.keys(), columns=splitting_variables)
x_test_spvar_df = pd.DataFrame(x_test_df.groupby(splitting_variables).groups.keys(), columns=splitting_variables)
x_val_spvar_df = pd.DataFrame(x_val_df.groupby(splitting_variables).groups.keys(), columns=splitting_variables)

In [ ]:
display(Markdown("Comparison between train and test distributions of numerical variables (histograms)"))
validationlib.plots.doubleHistogram(
    x_train_spvar_df, 
    x_test_spvar_df, 
    xlabel='Input',
    x1label='Train',
    x2label='Test',
    multiPlotsKwargs = {"tight_layout":True}
)
plt.show()

display(Markdown("Comparison between train and test distributions of numerical variables (hypothesis tests: Kolmogorov-Smirnov and Anderson-Darling)"))
table_pval_num = validationlib.tests.dist.dist_similarity_table(
    x_train_spvar_df, 
    x_test_spvar_df,  
    title="p-values",
    tests=["KS", "AD"]
)
display(table_pval_num)

In [ ]:
display(Markdown("### Voxel Tesselation proximity method"))
from scipy.spatial import ConvexHull
from validationlib.misc.split_validation import voxel_tesselation_proximity_method

validation_result = voxel_tesselation_proximity_method(
    x_train_spvar_df,
    x_test_spvar_df,
    compute_chmp=True,
)

display(validation_result)

if len(splitting_variables) == 2:
    hull = ConvexHull(x_train_spvar_df[splitting_variables].values)

    offset = 0.005
    var0_offset = offset * (x_test_spvar_df[splitting_variables[0]].max() - x_test_spvar_df[splitting_variables[0]].min())
    var1_offset = offset * (x_test_spvar_df[splitting_variables[1]].max() - x_test_spvar_df[splitting_variables[1]].min())

    fig, axs = plt.subplots(figsize=(10,10))

    for i, simplex in enumerate(hull.simplices):
        label = "Training CH" if i == 0 else None
        axs.plot(x_train_spvar_df.loc[simplex, splitting_variables[0]], x_train_spvar_df.loc[simplex, splitting_variables[1]], 'k:', label=label)

    axs.scatter(x_train_spvar_df[splitting_variables[0]], x_train_spvar_df[splitting_variables[1]], s=10,facecolors='none', edgecolors='k', label='Train')
    # axs.scatter(x_test_spvar_df[splitting_variables[0]], x_test_spvar_df[splitting_variables[1]], s=10, c='k', label='Test')
    axs.scatter(x_test_spvar_df[splitting_variables[0]][validation_result.isolated_test], x_test_spvar_df[splitting_variables[1]][validation_result.isolated_test], s=60, marker="x", label='Isolated test', c="b", linewidths=1)
    axs.scatter(x_test_spvar_df[splitting_variables[0]][validation_result.test_outside_ch], x_test_spvar_df[splitting_variables[1]][validation_result.test_outside_ch], s=80, marker="+", label='Outside CH', c="b", linewidths=1)
    axs.scatter(x_test_spvar_df[splitting_variables[0]][validation_result.phacking_test], x_test_spvar_df[splitting_variables[1]][validation_result.phacking_test], s=60, marker="x", label='p-hacking', c="r", linewidths=1)
    axs.scatter(x_test_spvar_df[splitting_variables[0]][validation_result.valid_test_point], x_test_spvar_df[splitting_variables[1]][validation_result.valid_test_point], s=30, marker="o", label='Valid test', c="g", linewidths=1)

    for i in x_test_spvar_df.index:
        axs.text(x_test_spvar_df.loc[i, splitting_variables[0]] + var0_offset, x_test_spvar_df.loc[i, splitting_variables[1]] + var1_offset, str(i), fontsize=8)

    axs.set_xlabel(splitting_variables[0])
    axs.set_ylabel(splitting_variables[1])

    plt.legend(bbox_to_anchor=(0.5, 1.07), loc='upper center', borderaxespad=0., ncols=3)
    plt.tight_layout()
    plt.show()

---

## MODEL

Here, we plot the training curves of the model.

In [ ]:
training_curves_path = os.path.join(csv_dir, "training_curves.json")
if os.path.exists(training_curves_path):
    with open(training_curves_path, "r") as f:
        training_logs = json.load(f)

    training_metrics_dict = {"Loss": training_logs["train_loss"]}
    validation_metrics_dict = {"Loss": training_logs["test_loss"]}

    validationlib.models.validation.training_curves_plot(
        training_metrics_dict,
        validation_metrics_dict,
    )
    plt.show()
else:
    display(Markdown("*Training curves not available for this model.*"))

---

## ERROR QUANTIFICATION

In the following sections, we study the error of the model: its overall magnitude, its distributions (conditioned to inputs and outputs), if it presents biases, etc. For that we will use two metrics: the residue and the absolute error, defined for point $i$ as follows:

$$\text{Residue}_i = y_i - \hat{y}_i$$

$$\text{AbsErr}_i = |y_i - \hat{y}_i|$$

where $y_i$ and $\hat{y}_i$ are the ground truth and predicted $c_p$ respectively for point  $i$.

In [ ]:
# Definition of the metrics

residue_metric = validationlib.misc.metrics.DistanceMetrics("Residue").define_metric(lambda y_true, y_pred: y_true - y_pred)
abserr_metric = validationlib.misc.metrics.DistanceMetrics("Absolute Error").define_metric(lambda y_true, y_pred: np.abs(y_true - y_pred))

### Outside CH and potential p-hacking analysis


Here we perform two different comparisons:
 - We compare the error distribution of the test points considered as valid against the error distribution of test points flagged with risk of p-hacking. If those flagged with risk of p-hacking have truly p-hacking, their error should be significantly lower than the error of the points considered valid.
 - We compare the error distribution of the test points considered as valid against the error distribution of test points outside the training CH. Usually, test points outside the training CH will have a significantly higher error, since the model is extrapolating there.



In [ ]:
vtpm_masks_df = pd.DataFrame(index=x_test_df.index, columns=["phacking", "outside_ch", "valid"])

for i in x_test_spvar_df.index:
    spvar_mask = x_test_df[splitting_variables].apply(lambda row: (row == x_test_spvar_df.loc[i, splitting_variables]).all(), axis=1)

    vtpm_masks_df.loc[spvar_mask, "phacking"] = validation_result.phacking_test[i]
    vtpm_masks_df.loc[spvar_mask, "outside_ch"] = validation_result.test_outside_ch[i]
    vtpm_masks_df.loc[spvar_mask, "valid"] = validation_result.valid_test_point[i]
    vtpm_masks_df.loc[spvar_mask, "isolated"] = validation_result.isolated_test[i]

if vtpm_masks_df["phacking"].any():
    display(Markdown("### Comparison of the points with risk of p-hacking and the valid points"))
    display(Markdown("Comparison using histograms"))
    validationlib.plots.doubleHistogram(
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["valid"]], 
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["phacking"]], 
        xlabel=abserr_metric.name,
        x1label='Valid',
        x2label='p-hacking',
        bins=20,
        logscale=True,
    )
    plt.show()

    display(Markdown("Comparison using hypothesis tests (Kolmogorov-Smirnov and Anderson-Darling)"))
    displayed_table = validationlib.tests.dist.dist_similarity_table(
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["valid"]], 
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["phacking"]],
        title="Hypothesis Tests Results (p-value)",
        tests=["AD", "KS", "KW"]
    )
    display(displayed_table)
else:
    display(Markdown("#### No points with risk of p-hacking found in the test set."))

if vtpm_masks_df["outside_ch"].any():
    display(Markdown("### Comparison of the points outside the convex hull and the valid points"))
    display(Markdown("Comparison using histograms"))
    validationlib.plots.doubleHistogram(
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["valid"]], 
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["outside_ch"]], 
        xlabel=abserr_metric.name,
        x1label='Valid',
        x2label='Outside CH',
        bins=20,
        logscale=True
    )
    plt.show()

    display(Markdown("Comparison using hypothesis tests (Kolmogorov-Smirnov and Anderson-Darling"))
    displayed_table = validationlib.tests.dist.dist_similarity_table(
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["valid"]], 
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["outside_ch"]],
        title="Hypothesis Tests Results (p-value)",
        tests=["AD", "KS", "KW"]
    )
    display(displayed_table)
else:
    display(Markdown("#### No points outside the training convex hull found in the test set."))

if vtpm_masks_df["isolated"].any():
    display(Markdown("### Comparison of the isolated points and the valid points"))
    display(Markdown("Comparison using histograms"))
    validationlib.plots.doubleHistogram(
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["valid"]], 
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["isolated"]], 
        xlabel=abserr_metric.name,
        x1label='Valid',
        x2label='Isolated',
        bins=20,
        logscale=True
    )
    plt.show()

    display(Markdown("Comparison using hypothesis tests (Kolmogorov-Smirnov and Anderson-Darling"))
    displayed_table = validationlib.tests.dist.dist_similarity_table(
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["valid"]], 
        abserr_metric(yt_test_df, yh_test_df)[vtpm_masks_df["isolated"]],
        title="Hypothesis Tests Results (p-value)",
        tests=["AD", "KS", "KW"]
    )
    display(displayed_table)
else:
    display(Markdown("#### No points outside the training convex hull found in the test set."))

if vtpm_masks_df["phacking"].any() or vtpm_masks_df["outside_ch"].any() or vtpm_masks_df["isolated"].any():
    vtpm_masks_df_classes = pd.DataFrame(index=x_test_df.index, columns=["Kind"])
    vtpm_masks_df_classes.loc[vtpm_masks_df["phacking"], "Kind"] = "p-hacking"
    vtpm_masks_df_classes.loc[vtpm_masks_df["outside_ch"], "Kind"] = "Outside CH"
    vtpm_masks_df_classes.loc[vtpm_masks_df["valid"], "Kind"] = "Valid"
    vtpm_masks_df_classes.loc[vtpm_masks_df["isolated"], "Kind"] = "Isolated"

    validationlib.plots.boxplot(
        vtpm_masks_df_classes,
        abserr_metric(yt_test_df, yh_test_df),
        categorical=True,
        var_y = yt_test_df.columns[0], # Name of the output variable
        xlabel = 'Kind',
        ylabel = residue_metric.name,
        showfliers = False,
        # trimStds=3,
        multiPlotsKwargs = {'tight_layout': True}
    )
    plt.show()

    validationlib.plots.violinPlot(
        vtpm_masks_df_classes,
        abserr_metric(yt_test_df, yh_test_df),
        categorical=True,
        var_y = yt_test_df.columns[0], # Name of the output variable
        xlabel = 'Kind',
        ylabel = residue_metric.name,
        showextrema=True,
        # trimStds=3,
        multiPlotsKwargs = {'tight_layout': True}
    )
    plt.show()

### Pointwise error metrics

Here we study the general aspects of the model's error.

#### RESIDUE

In [ ]:
# Summary of metric's statistics. Desired statistics are given as a dictionary.
# Confidence intervals of specified statistics are calculated using the bootstrap method.
statistics = {
    "mean": np.mean,
    "median": [np.percentile, {"q": 50}],
    "std": np.std,
    "IQR": st.iqr,
    "kurtosis": st.kurtosis,
    "skewness": st.skew
}
displayed_table = validationlib.tables.summary.prediction_stats(
    residue_metric(yt_test_df, yh_test_df), # Metric from which the statistics are calculated
    statistics=statistics, # Dictionary of statistics to calculate
    precision=3, # Number of decimal places to display in the table
    nsim=100, # Number of simulations for the bootstrap method
    method='Bootstrap' # Method used to calculate the confidence intervals. 
)
display(displayed_table)

# Cumulative distribution of the metric (table)
statistics_percentile = {"%i%%" % i: [np.percentile, {"q": i}] for i in [1,5,10,50,90,95,99]} # Dict of percentiles to calculate
displayed_table = validationlib.tables.summary.prediction_stats(
    residue_metric(yt_test_df, yh_test_df),
    statistics=statistics_percentile, 
    precision=3, 
    nsim=100,
    method='Bootstrap', 
    countMinMax=False
)
display(displayed_table)

# Cumulative distribution of the metric (plot)
validationlib.plots.cumulative(
    residue_metric(yt_test_df, yh_test_df),
    xlabel=residue_metric.name, 
    bins=1000,
    quantiles=[0.1, 0.50, 0.90, 0.95, 0.99] # Quantiles to display in the plot
)
plt.show()

#### ABS. ERR.

In [ ]:
# Summary of metric's statistics. Desired statistics are given as a dictionary.
# Confidence intervals of specified statistics are calculated using the bootstrap method.
statistics = {
    "mean": np.mean,
    "median": [np.percentile, {"q": 50}],
    "std": np.std,
    "IQR": st.iqr,
    "kurtosis": st.kurtosis,
    "skewness": st.skew
}
displayed_table = validationlib.tables.summary.prediction_stats(
    abserr_metric(yt_test_df, yh_test_df), # Metric from which the statistics are calculated
    statistics=statistics, # Dictionary of statistics to calculate
    precision=3, # Number of decimal places to display in the table
    nsim=100, # Number of simulations for the bootstrap method
    method='Bootstrap' # Method used to calculate the confidence intervals. 
)
display(displayed_table)

# Cumulative distribution of the metric (table)
statistics_percentile = {"%i%%" % i: [np.percentile, {"q": i}] for i in [1,5,10,50,90,95,99]} # Dict of percentiles to calculate
displayed_table = validationlib.tables.summary.prediction_stats(
    abserr_metric(yt_test_df, yh_test_df),
    statistics=statistics_percentile, 
    precision=3, 
    nsim=100,
    method='Bootstrap', 
    countMinMax=False
)
display(displayed_table)

#### PREDICTED vs GROUND TRUTH

Comparison between the distributionss of true and predicted outputs. These distributions should be as close as possible.


In [ ]:
display(Markdown("Comparison using histograms"))
validationlib.plots.doubleHistogram(
    yt_test_df, 
    yh_test_df, 
    xlabel='Output',
    x1label='True',
    x2label='Predicted',
    logscale=True
)
plt.show()

display(Markdown("Comparison using hypothesis tests (Kolmogorov-Smirnov and Anderson-Darling)"))
displayed_table = validationlib.tests.dist.dist_similarity_table(
    yt_test_df,
    yh_test_df,
    title="Hypothesis Tests Results (p-value)",
    tests=["AD", "KS"]
)
display(displayed_table)

In [ ]:
display(Markdown("Predicted vs true scatterplot and 2D histogram. This plots should be as close to the identity line as possible."))

validationlib.plots.scatterplot(
    yt_test_df,
    yh_test_df,
    c=residue_metric(yt_test_df, yh_test_df), # The metric is used to color the points
    correlationAxis="fit", # This displays a linear fit and a "normalized" slope (should be as close to 1 as possible)
    fit_info="default",
    significant_figures=3,
    xlabel="True",
    ylabel="Predicted",
    clabel=residue_metric.name,
    cmapCenter=residue_metric.neutral, # The color map is centered around the neutral value of the metric (the value that is considered "good", e.g. 0 for Residue)
    trimStds=3 # This trims the color map to the range of values within 3 standard deviations of the mean
)
plt.show()

In [ ]:
validationlib.plots.hist2D(
    yt_test_df, 
    yh_test_df,
    bins=100,
    xlabel="True", 
    ylabel="Predicted",
    scale="frequency",
    correlationAxis="fit",
    logscale=True,
    figHsize=7,
    figAspectRatio=2
)
plt.show()

### P(E)

Here we study the marginal distribution of the error (not conditioned). For that we employ an AD test to compare the distribution of the error metric to some well known parametric distributions.

If a fit is detected, we further study the quality of said fit.

If the fit is determined to be appropriate, we can use the newly fit parametric distribution to characterize the model's error (e.g. outlier detection).

#### RESIDUE

In [ ]:
# Goodness of fit tests. These tests compare the distribution of the defined metric to a parametric distribution.
# This can be used with any parametric distribution from scipy.stats.

# This cell exectues the tests using a quick method
# Quick method has a large variability in the results for small to medium sample sizes (try running this cell multiple times to see how the results change)
# If this behavior is observed, it is recommended to use the robust method, which is more computationally expensive
displayed_table, df = validationlib.tables.summary.fit_distribution_similarity(
    residue_metric(yt_test_df, yh_test_df),
    test = 'AD', # Anderson-Darling test to compare the distributions
    method="quick",
    dist = validationlib.tests.dist.COMMON_DISTS, # List of commonly used distributions
    title="Fit to each output-metric (quick-method)",
    robust_kwargs={"n_mc_samples": 1000},
    plot_fits=True
)
display(displayed_table)

#### ABS. ERR.

In [ ]:
# Goodness of fit tests. These tests compare the distribution of the defined metric to a parametric distribution.
# This can be used with any parametric distribution from scipy.stats.

# This cell exectues the tests using a quick method
# Quick method has a large variability in the results for small to medium sample sizes (try running this cell multiple times to see how the results change)
# If this behavior is observed, it is recommended to use the robust method, which is more computationally expensive
displayed_table, df = validationlib.tables.summary.fit_distribution_similarity(
    abserr_metric(yt_test_df, yh_test_df),
    test = 'AD', # Anderson-Darling test to compare the distributions
    method="quick",
    dist = validationlib.tests.dist.COMMON_DISTS, # List of commonly used distributions
    title="Fit to each output-metric (quick-method)",
    robust_kwargs={"n_mc_samples": 1000},
    plot_fits=True
)
display(displayed_table)

### P(E|X)

Here we study the distribution of the error conditioned to the model's inputs.

#### RESIDUE

In [ ]:
for i, out_var in enumerate(output_names):
    validationlib.plots.scatterplot(
        x_test_df, 
        residue_metric(yt_test_df, yh_test_df)[[out_var]],
        c=residue_metric(yt_test_df, yh_test_df)[[out_var]],
        correlationAxis="fit", 
        xlabel="Input Variable", 
        ylabel=residue_metric.name,
        title_prefix=yt_test_df.columns[i],
        trimStds=3,
        clabel=residue_metric.name,
        multiPlotsKwargs = {'tight_layout': True}
    )
    plt.show()

In [ ]:
for i, out_var in enumerate(output_names):
    validationlib.plots.hist2D(
        x_test_df, 
        pd.concat([residue_metric(yt_test_df, yh_test_df)[[out_var]]]*len(input_names), axis=1),
        bins=100,
        xlabel="Input Variable", 
        ylabel=residue_metric.name,
        scale="frequency",
        correlationAxis="fit",
        logscale=True,
        figHsize=21,
        figAspectRatio=4.5
    )
    plt.show()

In [ ]:
display(Markdown("""##### Detection of trends in the **residue** metric against each numerical variable
This returns a table for each **input** variable, containing:
- The correlation coefficient
- The p-value of the trend test. The lower the p-value, the stronger the evidence against the null hypothesis that there is no trend.
- If we are testing for a linear trend (Pearson coefficient), it is also displayed the normalized slope of the linear trend. The higher the slope, the stronger the linear trend. Since it is normalized, the slope comprises the range -1 to 1.
"""))

display(Markdown("Pearson coefficient"))
lintrend_tables = validationlib.tests.bias.trend_table(x_test_df, residue_metric(yt_test_df, yh_test_df))
for table in lintrend_tables:
    display(table)
    
display(Markdown("Spearman coefficient"))
lintrend_tables = validationlib.tests.bias.trend_table(x_test_df, residue_metric(yt_test_df, yh_test_df), method="spearman")
for table in lintrend_tables:
    display(table)

To further study P(E|X), we bin each variable. The number of bins is determined by the Sturges rule.

In [ ]:
for i, out_var in enumerate(output_names):
    validationlib.plots.boxplot(
        x_test_df,
        residue_metric(yt_test_df, yh_test_df)[[out_var]],
        bins = "sturges", # Number of bins for discretization. These bins are equally spaced between the minimum and maximum values of the numerical variable
        var_y = yt_test_df.columns[i], # Name of the output variable
        xlabel = 'Bins',
        ylabel = residue_metric.name,
        showfliers = False,
        trimStds=3,
        multiPlotsKwargs = {'tight_layout': True}
    )
    plt.show()

    # Violin plot of the metric against each numerical variable
    validationlib.plots.violinPlot(
        x_test_df,
        residue_metric(yt_test_df, yh_test_df)[[out_var]],
        bins = "sturges", # Number of bins for discretization. These bins are equally spaced between the minimum and maximum values of the numerical variable
        var_y = yt_test_df.columns[0], # Name of the output variable
        xlabel = 'Bins',
        ylabel = residue_metric.name,
        showextrema = True,
        trimStds=3,
        multiPlotsKwargs = {'tight_layout': True}
    )
    plt.show()

We can perform some hypothesis test on the obtained bins to check if they present biases. Specifically, these tests are:
 - Levene. Null hyp. $\rightarrow$ all bins have the same variance.
 - ANOVA: Null hyp. $\rightarrow$ all bins have the same mean. (_NOTE: In order to have valid ANOVA results, the repective bins must be homocedastic and normally distributed. If these requirements are not fulfilled, ANOVA results are shown as `nan`_).
  - Kruskall-Wallis: Null hyp. $\rightarrow$ all bins come from the same distribution.

In [ ]:
# Create an auxiliary dataframe with the binned numerical variables
# This allows to use the tests and plots for categorical variables with numerical variables
df_binned = pd.DataFrame(index=x_test_df.index)
for i, in_var in enumerate(input_names):
    _, df_binned[in_var+"_binned"] = validationlib.misc.subsampling.bin_data(x_test_df[in_var], bins="sturges")

num_binned_vars = df_binned.columns
df_binned = df_binned.join(residue_metric(yt_test_df, yh_test_df))

In [ ]:
output_dict = validationlib.tests.bias.parametric_bias_quantification_pipeline(
    df_binned,
    num_binned_vars,
    yt_test_df.columns,
)

for key, val in output_dict.items():
    if val != []:
        display(val)

In [ ]:
table = validationlib.tests.bias.bias_detection_table(
    df_binned,
    num_binned_vars,
    yt_test_df.columns,
    method="kruskal"
)
display(table)

#### ABS. ERR.

In [ ]:
for i, out_var in enumerate(output_names):
    validationlib.plots.scatterplot(
        x_test_df, 
        abserr_metric(yt_test_df, yh_test_df)[[out_var]],
        c=abserr_metric(yt_test_df, yh_test_df)[[out_var]],
        correlationAxis="fit", 
        xlabel="Input Variable", 
        ylabel=abserr_metric.name,
        title_prefix=yt_test_df.columns[i],
        trimStds=3,
        clabel=abserr_metric.name,
        multiPlotsKwargs = {'tight_layout': True}
    )
    plt.show()

In [ ]:
for i, out_var in enumerate(output_names):
    validationlib.plots.hist2D(
        x_test_df, 
        pd.concat([abserr_metric(yt_test_df, yh_test_df)[[out_var]]]*len(input_names), axis=1),
        bins=100,
        xlabel="Input Variable", 
        ylabel=abserr_metric.name,
        scale="frequency",
        correlationAxis="fit",
        logscale=True,
        figHsize=21,
        figAspectRatio=4.5
    )
    plt.show()

In [ ]:
display(Markdown("""##### Detection of trends in the **absolute error** metric against each numerical variable
This returns a table for each **input** variable, containing:
- The correlation coefficient
- The p-value of the trend test. The lower the p-value, the stronger the evidence against the null hypothesis that there is no trend.
- If we are testing for a linear trend (Pearson coefficient), it is also displayed the normalized slope of the linear trend. The higher the slope, the stronger the linear trend. Since it is normalized, the slope comprises the range -1 to 1.
"""))

display(Markdown("Pearson coefficient"))
lintrend_tables = validationlib.tests.bias.trend_table(x_test_df, abserr_metric(yt_test_df, yh_test_df))
for table in lintrend_tables:
    display(table)

display(Markdown("Spearman coefficient"))
lintrend_tables = validationlib.tests.bias.trend_table(x_test_df, abserr_metric(yt_test_df, yh_test_df), method="spearman")
for table in lintrend_tables:
    display(table)

To further study P(E|X), we bin each variable. The number of bins is determined by the Sturges rule.

In [ ]:
for i, out_var in enumerate(output_names):
    validationlib.plots.boxplot(
        x_test_df,
        abserr_metric(yt_test_df, yh_test_df)[[out_var]],
        bins = "sturges", # Number of bins for discretization. These bins are equally spaced between the minimum and maximum values of the numerical variable
        var_y = yt_test_df.columns[i], # Name of the output variable
        xlabel = 'Bins',
        ylabel = abserr_metric.name,
        showfliers = False,
        trimStds=3,
        multiPlotsKwargs = {'tight_layout': True}
    )
    plt.show()

    # Violin plot of the metric against each numerical variable
    validationlib.plots.violinPlot(
        x_test_df,
        abserr_metric(yt_test_df, yh_test_df)[[out_var]],
        bins = "sturges", # Number of bins for discretization. These bins are equally spaced between the minimum and maximum values of the numerical variable
        var_y = yt_test_df.columns[0], # Name of the output variable
        xlabel = 'Bins',
        ylabel = abserr_metric.name,
        showextrema = True,
        trimStds=3,
        multiPlotsKwargs = {'tight_layout': True}
    )
    plt.show()

We can perform some hypothesis test on the obtained bins to check if they present biases. Specifically, these tests are:
 - Levene. Null hyp. $\rightarrow$ all bins have the same variance.
 - ANOVA: Null hyp. $\rightarrow$ all bins have the same mean. (_NOTE: In order to have valid ANOVA results, the repective bins must be homocedastic and normally distributed. If these requirements are not fulfilled, ANOVA results are shown as `nan`_).
  - Kruskall-Wallis: Null hyp. $\rightarrow$ all bins come from the same distribution.

In [ ]:
df_binned = pd.DataFrame(index=x_test_df.index)
for i, in_var in enumerate(input_names):
    _, df_binned[in_var+"_binned"] = validationlib.misc.subsampling.bin_data(x_test_df[in_var], bins="sturges")

num_binned_vars = df_binned.columns
df_binned = df_binned.join(abserr_metric(yt_test_df, yh_test_df))

In [ ]:
output_dict = validationlib.tests.bias.parametric_bias_quantification_pipeline(
    df_binned,
    num_binned_vars,
    yt_test_df.columns,
)

for key, val in output_dict.items():
    if val != []:
        display(val)

In [ ]:
table = validationlib.tests.bias.bias_detection_table(
    df_binned,
    num_binned_vars,
    yt_test_df.columns,
    method="kruskal"
)
display(table)

In [ ]:
if len(splitting_variables) == 2:
    display(Markdown("Since the splitting is done by two variables, we also show the error conditioned to the splitting variables"))

    for out_var in output_names:   
        grouped_df = pd.concat([x_test_df[splitting_variables], abserr_metric(yt_test_df, yh_test_df)], axis=1).groupby(splitting_variables)
        mean_error = grouped_df.mean().reset_index()
        q95_error = grouped_df.quantile(0.95).reset_index()
        fig, axs = plt.subplots(1, 2, figsize=(12, 6))
        h = axs[0].scatter(mean_error[splitting_variables[0]], mean_error[splitting_variables[1]], c=mean_error[out_var], cmap="viridis", s=100)
        plt.colorbar(h, ax=axs[0], label="Mean of " + abserr_metric.name)
        axs[0].set_xlabel(splitting_variables[0])
        axs[0].set_ylabel(splitting_variables[1])

        h = axs[1].scatter(q95_error[splitting_variables[0]], q95_error[splitting_variables[1]], c=q95_error[out_var], cmap="viridis", s=100)
        plt.colorbar(h, ax=axs[1], label="95th percentile of " + abserr_metric.name)
        axs[1].set_xlabel(splitting_variables[0])
        axs[1].set_ylabel(splitting_variables[1])

        fig.suptitle(f"Error conditioned to the splitting variables for output variable {out_var}")

        plt.tight_layout()
        plt.show()

### P(E|Y)

Here we study the distribution of the error conditioned to the output of the model

#### RESIDUE

In [ ]:
validationlib.plots.scatterplot(
    yt_test_df, 
    residue_metric(yt_test_df, yh_test_df),
    c=residue_metric(yt_test_df, yh_test_df),
    clabel=residue_metric.name,
    correlationAxis="fit", 
    xlabel="Output", 
    ylabel=residue_metric.name,
    trimStds=3,
    cmapCenter=residue_metric.neutral
)
plt.show()

In [ ]:
validationlib.plots.hist2D(
    yt_test_df, 
    residue_metric(yt_test_df, yh_test_df),
    bins=100,
    xlabel="Output Variable", 
    ylabel=abserr_metric.name,
    scale="frequency",
    correlationAxis="fit",
    logscale=True,
    figHsize=7,
    figAspectRatio=2
)
plt.show()

In [ ]:
display(Markdown("""##### Detection of trends in the **residue** metric against each numerical variable
This returns a table for each **output** variable, containing:
- The correlation coefficient
- The p-value of the trend test. The lower the p-value, the stronger the evidence against the null hypothesis that there is no trend.
- If we are testing for a linear trend (Pearson coefficient), it is also displayed the normalized slope of the linear trend. The higher the slope, the stronger the linear trend. Since it is normalized, the slope comprises the range -1 to 1.
"""))

display(Markdown("Pearson coefficient"))
lintrend_tables = validationlib.tests.bias.trend_table(yt_test_df, residue_metric(yt_test_df, yh_test_df))
for table in lintrend_tables:
    display(table)

display(Markdown("Spearman coefficient"))
lintrend_tables = validationlib.tests.bias.trend_table(yt_test_df, residue_metric(yt_test_df, yh_test_df), method="spearman")
for table in lintrend_tables:
    display(table)

To further study P(E|Y), we bin each output variable. The number of bins is determined by the Sturges rule.

In [ ]:
validationlib.plots.boxplot(
    yt_test_df,
    residue_metric(yt_test_df, yh_test_df),
    bins = "sturges",
    xlabel = 'Bins',
    ylabel = residue_metric.name,
    showfliers = True,
    trimStds=2, 
    multiPlotsKwargs = {'tight_layout': True}
)
plt.show()

validationlib.plots.violinPlot(
    yt_test_df,
    residue_metric(yt_test_df, yh_test_df),   
    bins = "sturges",
    xlabel = 'Bins',
    ylabel = residue_metric.name,
    showextrema = True,
    trimStds=2, 
    multiPlotsKwargs = {'tight_layout': True}
)
plt.show()

We can perform some hypothesis test on the obtained bins to check if they present biases. Specifically, these tests are:
 - Levene. Null hyp. $\rightarrow$ all bins have the same variance.
 - ANOVA: Null hyp. $\rightarrow$ all bins have the same mean. (_NOTE: In order to have valid ANOVA results, the repective bins must be homocedastic and normally distributed. If these requirements are not fulfilled, ANOVA results are shown as `nan`_).
  - Kruskall-Wallis: Null hyp. $\rightarrow$ all bins come from the same distribution.

In [ ]:
df_binned = pd.DataFrame(index=yt_test_df.index)
for i, out_var in enumerate(yt_test_df.columns):
    _, df_binned[out_var+"_binned"] = validationlib.misc.subsampling.bin_data(yt_test_df[out_var], bins="sturges")

num_binned_outputs = df_binned.columns
df_binned = df_binned.join(residue_metric(yt_test_df, yh_test_df))

In [ ]:
bias_quantification_output = validationlib.tests.bias.parametric_bias_quantification_pipeline(
    df_binned,
    num_binned_outputs,
    yt_test_df.columns,
    alpha=0.05,
    min_bin_size=20,
    min_normal_proportion=0.9,
    colorPair=['green', 'red'],
    cmap="viridis"
)

for key, value in bias_quantification_output.items():
    if isinstance(value, list):
        for i, table in enumerate(value):
            display(table.transpose())
    else:
        display(value)

In [ ]:
table = validationlib.tests.bias.bias_detection_table(
    df_binned,
    num_binned_outputs,
    yt_test_df.columns,
    method="kruskal"
)
display(table)

#### ABS. ERR.

In [ ]:
validationlib.plots.scatterplot(
    yt_test_df, 
    abserr_metric(yt_test_df, yh_test_df),
    c=abserr_metric(yt_test_df, yh_test_df),
    clabel=abserr_metric.name,
    correlationAxis="fit", 
    xlabel="Output", 
    ylabel=abserr_metric.name,
    trimStds=3,
    cmapCenter=abserr_metric.neutral
)
plt.show()

In [ ]:
validationlib.plots.hist2D(
    yt_test_df, 
    abserr_metric(yt_test_df, yh_test_df),
    bins=100,
    xlabel="Output variable", 
    ylabel=abserr_metric.name,
    correlationAxis="fit",
    scale="frequency",
    logscale=True,
    figHsize=7,
    figAspectRatio=2
)
plt.show()

In [ ]:
display(Markdown("""##### Detection of trends in the **absolute error** metric against each numerical variable
This returns a table for each **output** variable, containing:
- The correlation coefficient
- The p-value of the trend test. The lower the p-value, the stronger the evidence against the null hypothesis that there is no trend.
- If we are testing for a linear trend (Pearson coefficient), it is also displayed the normalized slope of the linear trend. The higher the slope, the stronger the linear trend. Since it is normalized, the slope comprises the range -1 to 1.
"""))

display(Markdown("Pearson coefficient"))
lintrend_tables = validationlib.tests.bias.trend_table(yt_test_df, abserr_metric(yt_test_df, yh_test_df))
for table in lintrend_tables:
    display(table)

display(Markdown("Spearman coefficient"))
lintrend_tables = validationlib.tests.bias.trend_table(yt_test_df, abserr_metric(yt_test_df, yh_test_df), method="spearman")
for table in lintrend_tables:
    display(table)

To further study P(E|Y), we bin each variable. The number of bins is determined by the Sturges rule.

In [ ]:
validationlib.plots.boxplot(
    yt_test_df,
    abserr_metric(yt_test_df, yh_test_df),
    bins = "sturges",
    xlabel = 'Bins',
    ylabel = abserr_metric.name,
    showfliers = True,
    trimStds=2, 
    multiPlotsKwargs = {'tight_layout': True}
)
plt.show()

validationlib.plots.violinPlot(
    yt_test_df,
    abserr_metric(yt_test_df, yh_test_df),   
    bins = "sturges",
    xlabel = 'Bins',
    ylabel = abserr_metric.name,
    showextrema = True,
    trimStds=2, 
    multiPlotsKwargs = {'tight_layout': True}
)
plt.show()

We can perform some hypothesis test on the obtained bins to check if they present biases. Specifically, these tests are:
 - Levene. Null hyp. $\rightarrow$ all bins have the same variance.
 - ANOVA: Null hyp. $\rightarrow$ all bins have the same mean. (_NOTE: In order to have valid ANOVA results, the repective bins must be homocedastic and normally distributed. If these requirements are not fulfilled, ANOVA results are shown as `nan`_).
  - Kruskall-Wallis: Null hyp. $\rightarrow$ all bins come from the same distribution.

In [ ]:
df_binned = pd.DataFrame(index=yt_test_df.index)
for i, out_var in enumerate(yt_test_df.columns):
    _, df_binned[out_var+"_binned"] = validationlib.misc.subsampling.bin_data(yt_test_df[out_var], bins="sturges")

num_binned_outputs = df_binned.columns
df_binned = df_binned.join(abserr_metric(yt_test_df, yh_test_df))

In [ ]:
bias_quantification_output = validationlib.tests.bias.parametric_bias_quantification_pipeline(
    df_binned,
    num_binned_outputs,
    yt_test_df.columns,
    alpha=0.05,
    min_bin_size=20,
    min_normal_proportion=0.9,
    colorPair=['green', 'red'],
    cmap="viridis"
)

for key, value in bias_quantification_output.items():
    if isinstance(value, list):
        for i, table in enumerate(value):
            display(table.transpose())
    else:
        display(value)

In [ ]:
table = validationlib.tests.bias.bias_detection_table(
    df_binned,
    num_binned_outputs,
    yt_test_df.columns,
    method="kruskal"
)
display(table)

## UNCERTAINTY

In this section we build global and local uncertainty models for the error of the model under study. These models would be used in the case of new (unseen) samples, giving an estimate of the model's error in the form of a confidence interval.

Uncertainty models are built from the validation split. These models, by construction, will provide a confidence interval for the model's error, and they will be tested in the test set (hold-out). An uncertainty model will be considered valid if the proportion of test points inside the predicted uncertainty interval (i.e., the coverage of the model) is close to the specified CI size during training.

In [ ]:
# Concat validation and test datasets for uncertainty models
from scipy.sparse import csr_matrix

yt_valtest_df = pd.concat([yt_val_df, yt_test_df], axis=0, ignore_index=True)
yh_valtest_df = pd.concat([yh_val_df, yh_test_df], axis=0, ignore_index=True)
x_valtest_df = pd.concat([x_val_df, x_test_df], axis=0, ignore_index=True)

val_mask = np.zeros(yt_valtest_df.shape, dtype=bool)
val_mask[:len(yt_val_df)] = True
val_mask = csr_matrix(val_mask)

test_mask = np.zeros(yt_valtest_df.shape, dtype=bool)
test_mask[len(yt_val_df):] = True
test_mask = csr_matrix(test_mask)

In [ ]:
n_cal = val_mask.shape[0] - int(val_mask.nnz / val_mask.shape[1]) if hasattr(val_mask, 'nnz') else int((val_mask).sum())

# Adapt uncertainty model parameters to calibration set size
n_cal = yt_val_df.shape[0]  # val set is the calibration split
min_elems = max(10, n_cal // 10)         # at least 10, at most n_cal/10 per bin
n_bins_local = max(2, min(10, n_cal // max(10, min_elems)))  # bins × min_elems ≤ n_cal

ci_type = "right_tailed"
percentile_range = 95

print(f"Calibration set size: {n_cal}")
print(f"min_elems: {min_elems}  |  local bins: {n_bins_local}")

#### Global Uncertainty Model

In [ ]:
# Global uncertainty model
uncert_model_global = validationlib.tests.interval.BinnedUncertaintyModel(
    percentile_range=percentile_range,
    model_type="eqspaced_bins",
    ci_type=ci_type,
    method_kwargs={"bins":1, "nsim":100, "min_elems": min_elems, "conf": 0.95},
    matched_analysis=True
)
uncert_model_global.train(
    yh_valtest_df,
    abserr_metric(yt_valtest_df, yh_valtest_df),
    val_mask, # We can give a mask of the validation data to train the model
    x_variables=yt_test_df.columns,
    y_variables=yt_test_df.columns
)

# If we have a split of data, different than the one used to train the uncertainty model, we can create a ModelCoverage object to compute the coverage of the model.
cov_model = validationlib.tests.interval.ModelCoverage(matched_analysis=True, conf=0.95)
coverage_df, coverageci_df = cov_model.compute_coverage(
    yh_valtest_df,
    abserr_metric(yt_valtest_df, yh_valtest_df),
    test_mask, # The coverage is computed for the test data (as this split has not been used yet)
    uncert_model_global, # Uncertainty model
    x_variables=yt_test_df.columns,
    y_variables=yt_test_df.columns,
)

# We can complete the scatterplot with another split of data (e.g. the test data), and the coverage model
uncert_model_global.coverage_plot(
    yh_valtest_df,
    abserr_metric(yt_valtest_df, yh_valtest_df),
    val_mask,
    test_mask,
    cov_model,
    y_label=abserr_metric.name,
    plot_type="histogram",
    hist_data="test",
    bins=500,
    logscale=True
)
plt.show()

In [ ]:
tables = uncert_model_global.get_tables()
for table in tables:
    display(table)

#### Local Uncertainty Model. Output Conditioned

In [ ]:
# Uncertainty model output
uncert_model_localoutput = validationlib.tests.interval.BinnedUncertaintyModel(
    percentile_range=percentile_range,
    model_type="eqspaced_bins",
    ci_type=ci_type,
    method_kwargs={"bins":n_bins_local, "nsim":100, "min_elems": min_elems, "conf": 0.95},
    matched_analysis=True
)
uncert_model_localoutput.train(
    yh_valtest_df,
    abserr_metric(yt_valtest_df, yh_valtest_df),
    val_mask,
    x_variables=yt_test_df.columns,
    y_variables=yt_test_df.columns
)

cov_model = validationlib.tests.interval.ModelCoverage(matched_analysis=True, conf=0.95)
coverage_df, coverageci_df = cov_model.compute_coverage(
    yh_valtest_df,
    abserr_metric(yt_valtest_df, yh_valtest_df),
    test_mask,
    uncert_model_localoutput,
    x_variables=yt_test_df.columns,
    y_variables=yt_test_df.columns,
)

uncert_model_localoutput.coverage_plot(
    yh_valtest_df,
    abserr_metric(yt_valtest_df, yh_valtest_df),
    val_mask,
    test_mask,
    cov_model,
    y_label=abserr_metric.name,
    plot_type="histogram",
    hist_data="test",
    bins=500,
    logscale=True
)
plt.show()

In [ ]:
tables = uncert_model_localoutput.get_tables()
for table in tables:
    display(table)

#### Local Uncertainty Model. Input Conditioned

In [ ]:
# Uncertainty model numerical inputs
uncert_model_localinput_num = validationlib.tests.interval.BinnedUncertaintyModel(
    percentile_range=percentile_range,
    model_type="eqspaced_bins",
    ci_type=ci_type,
    method_kwargs={"bins":n_bins_local, "nsim":100, "min_elems": min_elems, "conf": 0.95},
    matched_analysis=False
)
uncert_model_localinput_num.train(
    x_valtest_df,
    abserr_metric(yt_valtest_df, yh_valtest_df),
    val_mask,
    x_variables=x_test_df.columns,
    y_variables=yt_test_df.columns
)

cov_modelnum = validationlib.tests.interval.ModelCoverage(matched_analysis=False, conf=0.95)
coverage_df, coverageci_df = cov_modelnum.compute_coverage(
    x_valtest_df,
    abserr_metric(yt_valtest_df, yh_valtest_df),
    test_mask,
    uncert_model_localinput_num,
    x_variables=x_test_df.columns,
    y_variables=yt_test_df.columns,
)

uncert_model_localinput_num.coverage_plot(
    x_valtest_df,
    abserr_metric(yt_valtest_df, yh_valtest_df),
    val_mask,
    test_mask,
    cov_modelnum,
    y_label=abserr_metric.name,
    plot_type="histogram",
    hist_data="test",
    bins=500,
    logscale=True
)
plt.show()

In [ ]:
tables = uncert_model_localinput_num.get_tables()
for table in tables:
    display(table)

#### Full Uncertainty Model

We can also combine all the previous uncertainty models in a single uncert. model. This model will output, given a sample, the biggest possible confidence interval from the output of all models.

Similarly to what we have done before, we can check the performance of this model by computing its coverage:

In [ ]:
# We can combine all the uncertainty models (global, output, numerical inputs and categorical inputs) in a single model
# This model will output, given a sample, the biggest possible confidence interval from the output of all models

# Compute predictions of all models
full_uncert_model = validationlib.tests.interval.CombinedUncertaintyModel(
    uncert_model_global, uncert_model_localoutput, uncert_model_localinput_num
)

full_uncert_preds = full_uncert_model.predict(
    x_valtest_df.join(yh_valtest_df),
    test_mask,
    x_variables=yt_test_df.columns,
    y_variables=yt_test_df.columns
)

# Coverage check
cov_model_full = validationlib.tests.interval.ModelCoverage(matched_analysis=True, conf=0.95)
coverage_df, coverageci_df = cov_model_full.compute_coverage(
    yh_valtest_df,
    abserr_metric(yt_valtest_df, yh_valtest_df),
    test_mask,
    model_predictions=full_uncert_preds,
    x_variables=yt_test_df.columns,
    y_variables=yt_test_df.columns,
)

print("Coverage of full model (by output variable):")
for out_var in yt_test_df.columns:
    print(f"{out_var}: {coverage_df.loc[out_var,out_var]}; CI95: {coverageci_df.loc[out_var,out_var]}")